In [1]:
import os
import sys
from pyspark.sql import SparkSession

In [2]:
# 1. 커널 재시작 후 가장 먼저 실행하세요!
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [3]:
# 기존 세션이 있다면 종료
try:
    spark.stop()
except:
    pass

In [4]:
os.environ["JAVA_HOME"] = "C:/tools/jdk17" # 설치하신 17 경로
os.environ["HADOOP_HOME"] = "C:/tools/hadoop-3.2.0"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin;" + os.environ["PATH"]

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "true") \
    .getOrCreate()

print("성공! 이제 Spark를 사용할 수 있습니다.")

성공! 이제 Spark를 사용할 수 있습니다.


In [6]:
import requests
import gzip
import shutil

In [26]:
# 1. CSV 압축 파일 다운로드
url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz"
file_gz = "fhvhv_tripdata_2021-01.csv.gz"
file_csv = "fhvhv_tripdata_2021-01.csv"

In [27]:
print("다운로드 시작... (시간이 조금 걸릴 수 있습니다)")
response = requests.get(url, stream=True)
with open(file_gz, 'wb') as f:
    f.write(response.content)

다운로드 시작... (시간이 조금 걸릴 수 있습니다)


In [28]:
print("압축 푸는 중...")
with gzip.open(file_gz, 'rb') as f_in:
    with open(file_csv, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print(f"완료! {file_csv} 파일이 준비되었습니다.")

압축 푸는 중...
완료! fhvhv_tripdata_2021-01.csv 파일이 준비되었습니다.


In [29]:
!head -n 101 fhvhv_tripdata_2021-01.csv > head.csv

In [24]:
import pandas as pd

In [25]:
df_pandas = pd.read_csv('head.csv')

In [26]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [27]:
# Pandas를 거치지 않고 Spark가 직접 CSV를 읽습니다. 
# 이 방식은 파이썬 버전 호환성 문제에서 훨씬 자유롭습니다.
df_spark = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('head.csv')

In [28]:
# 확인
df_spark.show()
df_spark.printSchema()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [29]:
df_spark.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [30]:
from pyspark.sql import types

In [31]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [32]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [33]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [34]:
df = df.repartition(24)

In [35]:
import shutil
import os
if os.path.exists('fhvhv/2021/01/'):
    shutil.rmtree('fhvhv/2021/01/')

In [36]:
df.write.parquet('fhvhv/2021/01/')

In [77]:
# 폴더 안에 파일들이 생겼는지 확인
path = 'fhvhv/2021/01/'
if os.path.exists(path):
    print(f"폴더 존재함: {path}")
    print("생성된 파일 목록:", os.listdir(path))
else:
    print("아직 폴더가 생성되지 않았습니다.")

폴더 존재함: fhvhv/2021/01/
생성된 파일 목록: ['.part-00000-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00001-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00002-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00003-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00004-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00005-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00006-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00007-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00008-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00009-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00010-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00011-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c000.snappy.parquet.crc', '.part-00012-9c75d035-27eb-4a1b-9c60-18f67ac1b966-c00